In [ ]:
import polars as pl
import numpy as np
import sakura
import altair as alt

In [ ]:
temps = (
    pl.read_csv("../data/noaa.csv", try_parse_dates=True)
    .filter(pl.col("variable") == pl.lit("TMIN"))
    .with_columns(year=pl.col("date").dt.year())
)

complete_years = (
    temps.group_by("year").agg(pl.len()).filter(pl.col("len") >= 365).get_column("year")
).to_list()

temps = (
    temps.filter(pl.col("year").is_in(complete_years))
    .with_columns(t=(pl.col("date") - pl.date(pl.col("year"), 1, 1)).dt.total_days())
    # exclude leap days
    .filter(pl.col("t").is_between(0, 364))
    .select(["year", "t", "temperature"])
    .pivot(on="t", index="year")
    .sort("year")
)

blooms = (
    pl.read_csv("../data/bloom_dates.csv", try_parse_dates=True)
    .filter(pl.col("year").is_in(complete_years))
    .with_columns(t=(pl.col("date") - pl.date(pl.col("year"), 1, 1)).dt.total_days())
    .sort("year")
)

assert blooms.height == temps.height
assert (blooms["year"] == temps["year"]).all()

blooms = blooms.get_column("t").to_numpy()
temps = temps.drop("year").to_numpy()

In [ ]:
model = sakura.Schumacher(temps=temps, blooms=blooms)
model.fit()

In [ ]:
pred = model.predictive()(rng_key=model.key, temps=model.temps, blooms=None)["obs"]

In [ ]:
pred_data = (
    pl.DataFrame(np.array(pred).T, schema=[str(i) for i in range(250)])
    .with_row_index("data_id")
    .unpivot(index="data_id", variable_name="sample_id")
    .with_columns(
        pl.col("value").cast(pl.Float64),
        pl.col("sample_id").cast(pl.Int64),
        type=pl.lit("pred"),
    )
    .filter(pl.col("sample_id") <= 10)
    .select(["type", "data_id", "sample_id", "value"])
)

obs_data = (
    pl.DataFrame({"value": model.blooms})
    .with_row_index("data_id")
    .with_columns(pl.col("value").cast(pl.Float64), type=pl.lit("obs"), sample_id=None)
    .select(["type", "data_id", "sample_id", "value"])
)

alt.Chart(pl.concat([pred_data, obs_data], how="vertical")).mark_point().encode(
    alt.X("data_id"), alt.Y("value"), alt.Color("type")
)

In [ ]:
diag_chart = (
    alt.Chart(pl.DataFrame({"x": 0, "x2": 120, "y": 0, "y2": 120}))
    .mark_rule()
    .encode(alt.X("x"), alt.Y("y"), alt.X2("x2"), alt.Y2("y2"))
)

data_chart = (
    alt.Chart(pred_data.join(obs_data, on="data_id", suffix="_obs"))
    .mark_point()
    .encode(alt.X("value_obs"), alt.Y("value"))
)

data_chart + diag_chart